# Fleet Allocation Optimization

## Public Transport Demand Planning — Kharagpur → Kolkata Corridor

This notebook converts predicted passenger demand into an operational
fleet allocation plan using mathematical optimization.

### Decision Objective

Determine the number of bus and rail services required on each corridor
segment while minimizing:

- Operating cost
- Unmet passenger demand

subject to fleet and service-capacity constraints.

In [1]:
import pandas as pd
import numpy as np
import pulp

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
# ============================================
# 3. LOAD DEMAND FORECAST
# ============================================

forecast_output = pd.read_csv(
    "../outputs/demand_forecast.csv"
)

forecast_output["date"] = pd.to_datetime(
    forecast_output["date"]
)

print("Forecast data loaded successfully.")
print("Shape:", forecast_output.shape)
print("\nDate range:")
print(
    forecast_output["date"].min(),
    "to",
    forecast_output["date"].max()
)

print("\nColumns:")
print(forecast_output.columns.tolist())

forecast_output.head()

Forecast data loaded successfully.
Shape: (128, 5)

Date range:
2025-10-14 00:00:00 to 2025-10-29 00:00:00

Columns:
['date', 'route', 'passengers', 'predicted_demand', 'forecast_error']


,date,route,passengers,predicted_demand,forecast_error
0,2025-10-14,Howrah_Kolkata_Bus,6794,6227,567
1,2025-10-15,Howrah_Kolkata_Bus,5529,6683,-1154
2,2025-10-16,Howrah_Kolkata_Bus,6601,6513,88
3,2025-10-17,Howrah_Kolkata_Bus,7549,6657,892
4,2025-10-18,Howrah_Kolkata_Bus,6539,5557,982


In [3]:
# ============================================
# 4. OPERATING ASSUMPTIONS
# ============================================

BUS_CAPACITY = 50
RAIL_CAPACITY = 1000

BUS_COST = 1
RAIL_COST = 8

MAX_BUS_SERVICES = 400
MAX_RAIL_SERVICES = 80

UNMET_DEMAND_PENALTY = 20

print("Operating assumptions loaded.")
print(f"Bus capacity       : {BUS_CAPACITY}")
print(f"Rail capacity      : {RAIL_CAPACITY}")
print(f"Bus cost/service   : {BUS_COST}")
print(f"Rail cost/service  : {RAIL_COST}")
print(f"Max bus services   : {MAX_BUS_SERVICES}")
print(f"Max rail services  : {MAX_RAIL_SERVICES}")
print(f"Unmet penalty      : {UNMET_DEMAND_PENALTY}")

Operating assumptions loaded.
Bus capacity       : 50
Rail capacity      : 1000
Bus cost/service   : 1
Rail cost/service  : 8
Max bus services   : 400
Max rail services  : 80
Unmet penalty      : 20


In [4]:
# ============================================
# 5. FLEET AVAILABILITY
# ============================================

TOTAL_BUS_FLEET = 40
TOTAL_RAIL_FLEET = 50

print("Fleet availability:")
print(f"Total buses available per day : {TOTAL_BUS_FLEET}")
print(f"Total rail services available : {TOTAL_RAIL_FLEET}")

Fleet availability:
Total buses available per day : 40
Total rail services available : 50


In [5]:
# ============================================
# 6. SELECT PLANNING DAY
# ============================================

planning_date = forecast_output["date"].min()

daily_forecast = forecast_output[
    forecast_output["date"] == planning_date
].copy()

print("Planning date:", planning_date.date())
print("\nForecast demand:")
print(
    daily_forecast[
        ["route", "predicted_demand"]
    ].to_string(index=False)
)

print(
    "\nTotal forecast demand:",
    round(daily_forecast["predicted_demand"].sum())
)

Planning date: 2025-10-14

Forecast demand:
                   route  predicted_demand
      Howrah_Kolkata_Bus              6227
     Howrah_Kolkata_Rail             14524
 Kharagpur_Midnapore_Bus              8462
Kharagpur_Midnapore_Rail             11205
  Midnapore_Uluberia_Bus              6568
 Midnapore_Uluberia_Rail             14519
     Uluberia_Howrah_Bus              6623
    Uluberia_Howrah_Rail             12661

Total forecast demand: 80789


In [6]:
# ============================================
# 7. FLEET-CONSTRAINED OPTIMIZATION MODEL
# ============================================

model = pulp.LpProblem(
    "Fleet_Allocation_Optimization",
    pulp.LpMinimize
)

routes = daily_forecast["route"].tolist()

# Decision variables
bus_services = {
    route: pulp.LpVariable(
        f"bus_{route}",
        lowBound=0,
        cat="Integer"
    )
    for route in routes
}

rail_services = {
    route: pulp.LpVariable(
        f"rail_{route}",
        lowBound=0,
        cat="Integer"
    )
    for route in routes
}

unmet_demand = {
    route: pulp.LpVariable(
        f"unmet_{route}",
        lowBound=0,
        cat="Continuous"
    )
    for route in routes
}

# --------------------------------------------
# Objective
# --------------------------------------------

model += pulp.lpSum(
    BUS_COST * bus_services[r]
    + RAIL_COST * rail_services[r]
    + UNMET_DEMAND_PENALTY * unmet_demand[r]
    for r in routes
)

# --------------------------------------------
# Route-level demand constraints
# --------------------------------------------

for _, row in daily_forecast.iterrows():

    route = row["route"]
    demand = row["predicted_demand"]

    model += (
        BUS_CAPACITY * bus_services[route]
        + RAIL_CAPACITY * rail_services[route]
        + unmet_demand[route]
        >= demand
    )

# --------------------------------------------
# Fleet-wide constraints
# --------------------------------------------

model += pulp.lpSum(
    bus_services[r] for r in routes
) <= TOTAL_BUS_FLEET

model += pulp.lpSum(
    rail_services[r] for r in routes
) <= TOTAL_RAIL_FLEET

# --------------------------------------------
# Solve
# --------------------------------------------

model.solve(
    pulp.PULP_CBC_CMD(msg=False)
)

print("Optimization status:")
print(pulp.LpStatus[model.status])

Optimization status:
Optimal


In [7]:
# ============================================
# 8. EXTRACT OPTIMAL FLEET ALLOCATION
# ============================================

optimization_results = []

for route in routes:

    demand = daily_forecast.loc[
        daily_forecast["route"] == route,
        "predicted_demand"
    ].iloc[0]

    buses = int(round(pulp.value(bus_services[route])))
    rail = int(round(pulp.value(rail_services[route])))
    unmet = pulp.value(unmet_demand[route])

    capacity = (
        BUS_CAPACITY * buses
        + RAIL_CAPACITY * rail
    )

    optimization_results.append({
        "route": route,
        "forecast_demand": round(demand),
        "bus_services": buses,
        "rail_services": rail,
        "capacity_provided": capacity,
        "unmet_demand": round(unmet, 2),
        "utilization_pct": round(
            demand / capacity * 100, 2
        ) if capacity > 0 else 0
    })

optimization_results = pd.DataFrame(
    optimization_results
)

optimization_results

,route,forecast_demand,bus_services,rail_services,capacity_provided,unmet_demand,utilization_pct
0,Howrah_Kolkata_Bus,6227,4,6,6200,27.0,100.44
1,Howrah_Kolkata_Rail,14524,2,0,100,14424.0,14524.00
2,Kharagpur_Midnapore_Bus,8462,0,0,0,8462.0,0.00
3,Kharagpur_Midnapore_Rail,11205,0,6,6000,5205.0,186.75
4,Midnapore_Uluberia_Bus,6568,11,6,6550,18.0,100.27
5,Midnapore_Uluberia_Rail,14519,10,14,14500,19.0,100.13
6,Uluberia_Howrah_Bus,6623,0,6,6000,623.0,110.38
7,Uluberia_Howrah_Rail,12661,13,12,12650,11.0,100.09


In [8]:
# ============================================
# 9. NETWORK-LEVEL SUMMARY
# ============================================

total_buses_used = optimization_results["bus_services"].sum()
total_rail_used = optimization_results["rail_services"].sum()
total_unmet = optimization_results["unmet_demand"].sum()

total_cost = (
    total_buses_used * BUS_COST
    + total_rail_used * RAIL_COST
    + total_unmet * UNMET_DEMAND_PENALTY
)

print("NETWORK OPTIMIZATION SUMMARY")
print("-----------------------------------")
print(f"Buses used        : {total_buses_used}")
print(f"Buses available   : {TOTAL_BUS_FLEET}")
print(f"Rail services used: {total_rail_used}")
print(f"Rail available    : {TOTAL_RAIL_FLEET}")
print(f"Total unmet demand: {total_unmet:.2f}")
print(f"Total operating cost: {total_cost:.2f}")

NETWORK OPTIMIZATION SUMMARY
-----------------------------------
Buses used        : 40
Buses available   : 40
Rail services used: 50
Rail available    : 50
Total unmet demand: 28789.00
Total operating cost: 576220.00
